# SpaceX Launch Analysis with SQL

This notebook explores the historical SpaceX launch dataset using SQL. It is designed to run locally in Jupyter or Google Colab without a cloud database account, credentials, or SQL magic extensions.

**Tools:** Python, pandas, SQLite, SQL

The notebook uses the repository's `Spacex.csv` file and creates a temporary SQLite database in memory. The source CSV is not modified.

## 1. Load the data and create a local SQL table

Make sure `Spacex.csv` is in the same folder as this notebook. If running in Colab, upload the CSV or clone/download the repository first.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

csv_path = Path("Spacex.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Spacex.csv was not found. Place it beside this notebook, "
        "or update csv_path to the file's location."
    )

df = pd.read_csv(csv_path)
df.columns = [
    "Date", "TimeUTC", "BoosterVersion", "LaunchSite", "Payload",
    "PayloadMassKG", "Orbit", "Customer", "MissionOutcome", "LandingOutcome"
]
df["PayloadMassKG"] = pd.to_numeric(df["PayloadMassKG"], errors="coerce")

# In-memory SQLite database: no server, password, or external service required.
conn = sqlite3.connect(":memory:")
df.to_sql("SpaceX", conn, if_exists="replace", index=False)

print(f"Loaded {len(df)} launch records into SQLite table SpaceX.")
display(df.head())

## 2. Explore the table


In [ ]:
pd.read_sql_query("SELECT * FROM SpaceX LIMIT 10;", conn)

## 3. SQL analysis tasks

Each section states the business question, runs a SQL query, and displays the result as a pandas DataFrame.

### Task 1 — What launch sites appear in the dataset?


In [ ]:
query = """
SELECT DISTINCT LaunchSite
FROM SpaceX
ORDER BY LaunchSite;
"""
pd.read_sql_query(query, conn)

### Task 2 — Which launches took place at sites beginning with `CCA`?


In [ ]:
query = """
SELECT Date, BoosterVersion, LaunchSite, Payload, PayloadMassKG, Orbit
FROM SpaceX
WHERE LaunchSite LIKE 'CCA%'
ORDER BY Date
LIMIT 10;
"""
pd.read_sql_query(query, conn)

### Task 3 — What is the total payload mass for NASA CRS missions?


In [ ]:
query = """
SELECT SUM(PayloadMassKG) AS TotalPayloadMassKG
FROM SpaceX
WHERE Customer LIKE '%NASA (CRS)%';
"""
pd.read_sql_query(query, conn)

### Task 4 — What is the average payload mass for the F9 v1.1 booster version?


In [ ]:
query = """
SELECT ROUND(AVG(PayloadMassKG), 2) AS AveragePayloadMassKG
FROM SpaceX
WHERE BoosterVersion LIKE 'F9 v1.1%';
"""
pd.read_sql_query(query, conn)

### Task 5 — What is the earliest recorded successful ground-pad landing?


In [ ]:
query = """
SELECT MIN(Date) AS FirstSuccessfulGroundPadLandingDate
FROM SpaceX
WHERE LandingOutcome LIKE '%Success%ground pad%';
"""
pd.read_sql_query(query, conn)

### Task 6 — Which booster versions had successful drone-ship landings with payloads between 4,000 and 6,000 kg?


In [ ]:
query = """
SELECT BoosterVersion, PayloadMassKG, LandingOutcome
FROM SpaceX
WHERE LandingOutcome LIKE '%Success%drone ship%'
  AND PayloadMassKG BETWEEN 4000 AND 6000
ORDER BY PayloadMassKG;
"""
pd.read_sql_query(query, conn)

### Task 7 — How many missions succeeded or failed?


In [ ]:
query = """
SELECT MissionOutcome, COUNT(*) AS MissionCount
FROM SpaceX
GROUP BY MissionOutcome
ORDER BY MissionCount DESC;
"""
pd.read_sql_query(query, conn)

### Task 8 — Which booster versions carried the maximum payload mass?


In [ ]:
query = """
SELECT BoosterVersion, PayloadMassKG
FROM SpaceX
WHERE PayloadMassKG = (SELECT MAX(PayloadMassKG) FROM SpaceX)
ORDER BY BoosterVersion;
"""
pd.read_sql_query(query, conn)

### Task 9 — Which launches in 2015 had a drone-ship landing failure?


In [ ]:
query = """
SELECT Date, BoosterVersion, LaunchSite, LandingOutcome
FROM SpaceX
WHERE LandingOutcome LIKE '%Failure%drone ship%'
  AND Date BETWEEN '2015-01-01' AND '2015-12-31'
ORDER BY Date;
"""
pd.read_sql_query(query, conn)

### Task 10 — How were landing outcomes distributed during the selected study period?


In [ ]:
query = """
SELECT LandingOutcome, COUNT(*) AS OutcomeCount
FROM SpaceX
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY LandingOutcome
ORDER BY OutcomeCount DESC;
"""
pd.read_sql_query(query, conn)

## 4. Data quality checks

Check missing values and date coverage before drawing conclusions. The launch records are historical, and findings describe this dataset rather than guaranteeing future landing outcomes.

In [ ]:
quality_checks = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1)
})
display(quality_checks)

pd.read_sql_query(
    "SELECT MIN(Date) AS FirstLaunchDate, MAX(Date) AS LastLaunchDate, COUNT(*) AS RecordCount FROM SpaceX;",
    conn
)

## Summary

This notebook demonstrates SQL filtering, pattern matching, aggregation, grouping, date filtering, and subqueries using a local SQLite database. It requires only Python, pandas, and the `Spacex.csv` file.

To rerun the analysis, restart the kernel and run the notebook from top to bottom. Close the in-memory database when finished.

In [ ]:
conn.close()
print("SQLite connection closed.")